In [0]:

# Create a DataFrame with a negative value and an implausibly large one
from pyspark.sql.functions import col, when

data = [(1, "Alice", 55000), (2, "Ben", -5000), (3, "Cara", 62000), (4, "Dev", 5000000)]
df = spark.createDataFrame(data, ["id", "name", "salary"])
df.display()

In [0]:
#Range check with comparison operators
invalid_v1 = df.filter((col("salary") < 0) | (col("salary") > 1000000))
invalid_v1.display()

In [0]:
#The same check with between()
invalid_v2 = df.filter(~col("salary").between(0, 1000000))
invalid_v2.display()

In [0]:
#Add a deviation column
df_dev = df.withColumn(
    "salary_deviation",
    when(col("salary") < 0, col("salary"))
    .when(col("salary") > 1000000, col("salary") - 1000000)
    .otherwise(0)
)
df_dev.filter(col("salary_deviation") != 0).display()

In [0]:
# Measure and report
total = df.count()
failures = df.filter(~col("salary").between(0, 1000000)).count()
print(f"Out-of-range salaries: {failures} out of {total} ({failures/total*100:.1f}%)")